In [1]:
# ==============================================================
# Notebook 6: Marked Hawkes Process (Magnitude Dependent)
#
# Goal:
# - Include earthquake magnitude as a mark
# - Fit magnitude-dependent triggering
#
# Model:
#
# lambda(t) =
# mu +
# sum(K * exp(a*(M_i-M0))
#     * exp(-beta*(t-ti)))
#
# ==============================================================


# --------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize


# --------------------------------------------------------------
# 2. Load data
# --------------------------------------------------------------

df = pd.read_csv(
    "../data/earthquakes_greece_2016_2026.csv"
)

df["time"] = pd.to_datetime(df["time"])

df = (
    df
    .sort_values("time")
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 3. Prepare times and magnitudes
# --------------------------------------------------------------

t0 = df["time"].iloc[0]

times = (
    df["time"] - t0
).dt.total_seconds() / (24*3600)

times = times.values


magnitudes = df["mag"].values


T = times[-1]


M0 = 4.5   # reference magnitude


print("Events:", len(times))


# --------------------------------------------------------------
# 4. Marked Hawkes intensity
# --------------------------------------------------------------

def marked_intensity(
    t,
    history_times,
    history_mag,
    mu,
    K,
    a,
    beta
):

    if len(history_times) == 0:
        return mu


    excitation = np.sum(
        K
        *
        np.exp(
            a*(history_mag-M0)
        )
        *
        np.exp(
            -beta*(t-history_times)
        )
    )


    return mu + excitation



# --------------------------------------------------------------
# 5. Log likelihood
# --------------------------------------------------------------

def marked_hawkes_loglik(
    params,
    times,
    magnitudes
):

    mu, K, a, beta = params


    if (
        mu <= 0
        or K < 0
        or a < 0
        or beta <= 0
    ):
        return -np.inf



    # Event contribution

    log_sum = 0


    for i,t in enumerate(times):

        lam = marked_intensity(
            t,
            times[:i],
            magnitudes[:i],
            mu,
            K,
            a,
            beta
        )

        log_sum += np.log(lam)



    # Integral contribution

    integral = mu*T


    for i in range(len(times)):

        integral += (
            K
            *
            np.exp(
                a*(magnitudes[i]-M0)
            )
            /
            beta
            *
            (
                1
                -
                np.exp(
                    -beta*(T-times[i])
                )
            )
        )


    return log_sum-integral



# --------------------------------------------------------------
# 6. Fit model
# --------------------------------------------------------------

def objective(params):

    return -marked_hawkes_loglik(
        params,
        times,
        magnitudes
    )



initial_guess = [
    0.2,  # mu
    0.5,  # K
    0.5,  # a
    1.0   # beta
]


result = minimize(
    objective,
    initial_guess,
    method="Nelder-Mead"
)



mu_hat, K_hat, a_hat, beta_hat = result.x



print("""
Fitted parameters
-----------------
""")

print("mu   =", mu_hat)
print("K    =", K_hat)
print("a    =", a_hat)
print("beta =", beta_hat)


print(
    "Log likelihood:",
    -result.fun
)


# --------------------------------------------------------------
# 7. Compare with previous Hawkes
# --------------------------------------------------------------

print("""
Interpretation:

mu:
    background seismic rate

K:
    base triggering strength

a:
    magnitude influence

beta:
    decay of aftershock influence

Large a means large earthquakes create
disproportionately more aftershocks.
""")


# --------------------------------------------------------------
# 8. Save parameters
# --------------------------------------------------------------

np.save(
    "../data/marked_hawkes_params.npy",
    np.array(
        [
            mu_hat,
            K_hat,
            a_hat,
            beta_hat
        ]
    )
)

Events: 2325

Fitted parameters
-----------------

mu   = 0.3073403153821932
K    = 0.913597360159584
a    = 1.3283648314312395
beta = 1.748655122782874
Log likelihood: -1734.91105753809

Interpretation:

mu:
    background seismic rate

K:
    base triggering strength

a:
    magnitude influence

beta:
    decay of aftershock influence

Large a means large earthquakes create
disproportionately more aftershocks.



In [2]:
# --------------------------------------------------------------
# 9. Compare AIC / BIC with previous models
# --------------------------------------------------------------

N = len(times)


# Log likelihoods from previous notebooks

poisson_ll = -3371.43

hawkes_ll = -1816.50

marked_hawkes_ll = -result.fun


# Number of parameters

k_poisson = 1

k_hawkes = 3

k_marked_hawkes = 4



# AIC = 2k - 2log(L)

aic_poisson = (
    2*k_poisson
    -
    2*poisson_ll
)

aic_hawkes = (
    2*k_hawkes
    -
    2*hawkes_ll
)

aic_marked_hawkes = (
    2*k_marked_hawkes
    -
    2*marked_hawkes_ll
)



# BIC = k log(N) - 2log(L)

bic_poisson = (
    k_poisson*np.log(N)
    -
    2*poisson_ll
)

bic_hawkes = (
    k_hawkes*np.log(N)
    -
    2*hawkes_ll
)

bic_marked_hawkes = (
    k_marked_hawkes*np.log(N)
    -
    2*marked_hawkes_ll
)



print("""
Model comparison
================
""")

print("AIC")
print("----------------")
print(f"Poisson        : {aic_poisson:.2f}")
print(f"Hawkes         : {aic_hawkes:.2f}")
print(f"Marked Hawkes  : {aic_marked_hawkes:.2f}")


print("\nBIC")
print("----------------")
print(f"Poisson        : {bic_poisson:.2f}")
print(f"Hawkes         : {bic_hawkes:.2f}")
print(f"Marked Hawkes  : {bic_marked_hawkes:.2f}")


Model comparison

AIC
----------------
Poisson        : 6744.86
Hawkes         : 3639.00
Marked Hawkes  : 3477.82

BIC
----------------
Poisson        : 6750.61
Hawkes         : 3656.25
Marked Hawkes  : 3500.83
